# MSCAD multi-stage intrusion-detection pipeline

Step 1 (PCAPNG -> one flow CSV per capture) lives in
`convert_pcapng_to_csvflows.ipynb`. This notebook runs **Step 2** and **Step 3**,
which are now separate stages.

| Step | What it does | Reads | Writes |
|------|--------------|-------|--------|
| **2 - Data acquisition** | one DataFrame per capture, tagged and reported | `C:\MSCAD2022\CSVs` | `C:\MSCAD2022\outputs\step2` |
| **3 - Dataset merging** | concatenate, verify, save, inspect | the Step 2 frames | `C:\MSCAD2022\outputs\step3` |
| 4+ | preprocessing onward | the merged dataset | later |

**Every path is absolute and lives on the data drive.** Nothing is resolved
against the notebook's working directory, so the outputs land in the same place
no matter where Jupyter was started or where this code is checked out.

## Setup

In [ ]:
import sys
from pathlib import Path

# The notebooks live in PROJECT_ROOT\notebooks, so when Jupyter is started in
# that folder the project root is one level up.  Starting Jupyter in the project
# root instead also works.  Nothing is hard-coded, so this is correct whether the
# code sits in C:\MSCAD2022\Notebook or in a git clone.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

import config

print("project root :", PROJECT_ROOT)
print("kernel python:", sys.version.split()[0])
print("flow CSVs in :", config.CSV_OUTPUT_DIR)
print("merged out   :", config.MERGED_DIR)
print("step 2 out   :", config.STEP2_OUTPUT_DIR)
print("kaggle ref   :", config.KAGGLE_CSV_DIR,
      "(exists)" if config.KAGGLE_CSV_DIR.is_dir() else "(not present - section 13 is skipped)")
print("random seed  :", config.RANDOM_SEED)

---
# Step 2 - Data acquisition

Loads every `<capture>_flows.csv` from `config.CSV_INPUT_DIR`
(`C:\MSCAD2022\CSVs`) into one DataFrame per capture, each tagged with:

| Column | Meaning |
|---|---|
| `Source_Capture` | the file stem the row came from |
| `Orig_Row` | the row's position inside its own file |

Nothing is merged, cleaned or reordered here. If the folder is missing or holds
no matching CSVs, this **stops** with a message naming the folder. If a single
file fails to read, it is reported and the rest still load.

### 2.1 Load the per-capture CSVs

In [ ]:
from src.mscad import data_loading

frames, acquisition_report, failures = data_loading.acquire_captures()

print()
print("captures loaded:", list(frames))
print("failures       :", failures if failures else "none")

### 2.2 What each capture contains

One row per file: name, rows, columns, size, label counts and the first/last
timestamp. Saved to `C:\MSCAD2022\outputs\step2\acquisition_report.csv`.

In [ ]:
data_loading.show_table(acquisition_report)

---
# Step 3 - Dataset merging

Concatenates the Step 2 frames in `config.CAPTURE_ORDER`, sorts by
`Timestamp_dt` with a stable mergesort, saves to
`C:\MSCAD2022\outputs\step3\MSCAD_merged.csv` and then **verifies the file on
disk**. Values are never changed: `inf`, `NaN` and duplicate rows all survive.

### 3.1 Column consistency

Every capture must carry the same columns in the same order. A mismatch is
**reported in a table**, never silently filled -- `pd.concat` would quietly
insert `NaN` that looks like missing data later instead of like the bug it is.

In [ ]:
consistency, columns_ok = data_loading.check_column_consistency(frames)

assert columns_ok, "the captures do not share one column set -- see the table above"

### 3.2 Merge

In [ ]:
merged, _ = data_loading.merge_flow_csvs(frames=frames)

### 3.3 Merge checks - PASS/FAIL with numbers

Row totals, per-capture counts, every `Label` count, the column layout, and the
`NaN` / `inf` totals, each against the per-file numbers from Step 2.

In [ ]:
infos = [{"capture": c,
          "rows": len(f),
          "label_counts": f[config.LABEL_COLUMN].value_counts().to_dict(),
          "nan_cells": int(f[[c2 for c2 in config.KAGGLE_FEATURES if c2 in f.columns]]
                           .isna().to_numpy().sum()),
          "inf_cells": int(__import__("numpy").isinf(
              f[[c2 for c2 in config.KAGGLE_FEATURES if c2 in f.columns]]
              .to_numpy(dtype="float64")).sum()),
          "timestamp_format": None,
          "first_timestamp": f["Timestamp_dt"].min(),
          "last_timestamp": f["Timestamp_dt"].max()}
         for c, f in frames.items()]

checks, all_passed = data_loading.check_merge(merged, infos)

print()
print("ALL MERGE CHECKS PASSED:", all_passed)
assert all_passed, "the merge is not faithful -- see the FAIL rows above"

### 3.4 Save the merged dataset, and prove it is on disk

Writes `C:\MSCAD2022\outputs\step3\MSCAD_merged.csv`, creating the folder if
needed, then re-reads it and compares the row and column counts with the
in-memory frame. You get `[OK]` or a clear `[FAIL]` and the absolute path.

If the file already exists and `config.OVERWRITE_MERGED` is `False`, it is
**not** rewritten -- that is said plainly and the existing file is loaded and
verified instead.

In [ ]:
save_result = data_loading.save_merged(merged, checks=checks, infos=infos)

print()
print("status :", save_result["status"])
print("path   :", save_result["target"])
print("ok     :", save_result["ok"])
assert save_result["ok"], "the merged file is not on disk, or does not match"

### 3.5 Load the merged dataset back from disk

In [ ]:
dataset = data_loading.load_merged()

print("shape from disk:", dataset.shape)
print("matches the in-memory merge:", dataset.shape == merged.shape)

### 3.6 Step 3 report files

`label_counts.csv`, `source_capture_x_label.csv`, `label_counts.png` and
`summary.txt`, all in `C:\MSCAD2022\outputs\step3`.

In [ ]:
reports = data_loading.save_step3_reports(dataset)

### 3.7 Inspect the merged dataset

Shape and memory, `head(10)` / `tail(5)` with all columns, dtypes and null
counts, label counts, the `Source_Capture` x `Label` crosstab, per-capture
timestamps, protocols, `NaN` / `inf` per column, duplicate counts, `describe()`
with an `inf_count` column, the label chart, and the Kaggle comparison.

Everything **counts and describes** - nothing is dropped, filled or changed.

In [ ]:
summary = data_loading.summarize_dataset(dataset)

---
## What Step 4 has to deal with

1. **`inf` in the rate columns** for zero-duration flows -- correct arithmetic,
   but sklearn and Keras both reject it. Clip, impute or drop: an explicit choice.
2. **`NaN`** wherever section 9 is non-zero -- per-column decisions, not a
   blanket `fillna(0)`.
3. **Identical-feature rows** -- real observations in floods, scans and brute
   force, but a random split puts identical rows on both sides and inflates
   every metric. Group-aware splitting is the defence.
4. **Class imbalance** -- see the ratio in section 5. Accuracy will be useless;
   macro-F1 and per-class recall are the metrics to watch.
5. **The identifier columns are label leakage.** Attacker IPs decided the labels
   in Step 1, so `Src IP` / `Dst IP` must not be features -- but they must
   survive for the sequence mining and attack-state graph.

---
# Stage 2: Load & merge flow CSVs

**What this stage does**

Reads the six per-capture CSVs Step 1 wrote (`<stem>_flows.csv`, 73 columns:
6 flow identifiers + 66 MSCAD features + `Label`), checks each header, and
concatenates them into one dataset with three bookkeeping columns added:

| Added column | Meaning |
|---|---|
| `Source_Capture` | which capture the row came from, taken from the file name |
| `Orig_Row` | the row's position inside its own file, so any row can be traced back |
| `Timestamp_dt` | the `Timestamp` text parsed to a real datetime; the text column is kept |

76 columns in total. Rows are then put in **time order** with a stable sort, so
the merged file reads like the attack session it came from — which is what the
Stage 8 sequence mining and attack-state graph need.

**What this stage deliberately does NOT do**

No cleaning: `inf`, `-inf`, `NaN` and duplicate rows are all carried through
exactly as they are, and the class imbalance is left alone. No encoding, no
scaling, no train/test split, no feature selection, no windowing, no training.
Those are Stage 3 onward. Step 2's only job is to produce one faithful merged
dataset and describe it honestly.

---
# Step 4 - Preprocessing

Cleans, encodes and scales the **whole** merged dataset from Step 3.

**In scope:** inf and NaN handling, dropping zero-variance columns, separating
the non-feature columns, label encoding, a binary target, Protocol one-hot,
and scaling.

**Deliberately NOT in scope:** no train/test split, no class-imbalance handling
(no SMOTE, no resampling, no class weights), no feature selection, no
modelling. Those are later steps.

| | |
|---|---|
| Input | `C:\MSCAD2022\outputs\step3\MSCAD_merged.csv` |
| Output | `C:\MSCAD2022\outputs\step4\` |
| Settings | `INF_STRATEGY`, `SCALER`, `LABEL_MAP`, `OVERWRITE_STEP4` in `config.py` |

Duplicate rows are **counted, never dropped** - the sequence work in a later
step depends on the original row order, and row *i* of `features`, `labels` and
`metadata` always refers to the same flow.

### Step 4 settings

In [ ]:
from src.mscad import preprocessing

print("input      :", config.MERGED_CSV)
print("output     :", config.STEP4_OUTPUT_DIR)
print("INF_STRATEGY   :", config.INF_STRATEGY, "  ('cap' or 'nan_median')")
print("SCALER         :", config.SCALER, "  ('minmax' or 'standard')")
print("LABEL_MAP      :", config.LABEL_MAP or "{} (nothing renamed)")
print("BENIGN_LABELS  :", config.BENIGN_LABELS, "-> Label_binary 0")
print("OVERWRITE_STEP4:", config.OVERWRITE_STEP4)
print("metadata kept aside:", len(config.METADATA_COLUMNS), "columns")

## 4.1 Clean

Runs in this order, each reported in its own table:

| | |
|---|---|
| **a** | `inf` -> capped at the column's largest finite value, plus a `zero_duration` marker |
| **b** | `NaN` -> median of its own column (never a blanket zero) |
| **c** | constant / zero-variance columns dropped, **measured not assumed** |
| **d** | non-feature columns moved to `metadata`, keeping row order |
| **e** | duplicate feature rows **counted, not dropped** |
| **f** | exact label strings reported; `LABEL_MAP` applied only if set |

## 4.2 Encode

`Label` -> `Label_encoded` (LabelEncoder, mapping saved as JSON) and
`Label_binary` (0 for `Benign`/`Normal`, 1 otherwise - they stay separate
classes in the multi-class target). `Protocol` -> four one-hot columns, with
anything unexpected landing in `Protocol_other`.

## 4.3 Scale

`SCALER` over the numeric features only - one-hot and binary columns keep their
0/1 meaning. **Both the scaled and the unscaled cleaned features are saved**:
this scaler is fitted on the whole dataset, which is fine for inspection but
would leak test information once a split exists, so a later step can refit it
on the training split alone from `features_clean.csv`.

### Run Step 4

In [ ]:
step4 = preprocessing.run_step4()

### What came out

`features_clean.csv`, `features_scaled.csv`, `labels.csv`, `metadata.csv`,
`label_mapping.json`, `feature_list.json`, `preprocess_artifacts.joblib` and
`step4_report.csv`, all in `C:\MSCAD2022\outputs\step4`. Every CSV was
re-read from disk and its row count compared with the frame in memory.

In [ ]:
if step4.get("status") == "loaded_existing":
    print("Existing Step 4 outputs were loaded (config.OVERWRITE_STEP4 is False).")
    print("Set OVERWRITE_STEP4 = True to recompute them.")
else:
    preprocessing.show_table(step4["report"])
    print()
    print("all checks passed:", step4["ok"])

### Row alignment

Row *i* of the features, labels and metadata is the same flow. This is what the
Step 8 sequence mining and attack-state graph depend on.

In [ ]:
clean, labels, meta = (step4["features_clean"], step4["labels"], step4["metadata"]) \
    if "features_clean" in step4 else (None, None, None)

if clean is not None:
    print("features:", clean.shape, "| labels:", labels.shape, "| metadata:", meta.shape)
    preprocessing.show_table(
        meta.head(5).join(labels.head(5)[["Label", "Label_encoded", "Label_binary"]]))

## 4.4 Final preprocessed dataset (`preprocessing.csv`)

One file laid out for **reading** and for the sequence / attack-chain work, in
the original row order:

```
Src IP | Dst IP | Src Port | Dst Port | Protocol | Timestamp | <scaled features> | Label
```

- The six identifiers carry their **original** values from the Step 3 merged
  data: IP strings unchanged, ports as integers, `Timestamp` as the original
  text. None of them is scaled.
- `Protocol` is **one readable column** - `TCP`, `UDP`, `ICMP` or `OTHER` -
  derived from the four `Protocol_*` one-hot columns. Exactly one of the four is
  `1` in every row, so the mapping is lossless and reversible, and a row that
  does not have exactly one set is reported as `[FAIL]` rather than guessed at.
  The four one-hot columns themselves are **not** in this file.
- The last column is the original `Label` string. `Label_encoded` and
  `Label_binary` are **not** here; `labels.csv` still has all three.

> **This file is the readable view, not the training matrix.** Two reasons:
> the identifier columns leak the answer - the attacker IPs decided the labels
> back in Step 1, so a model trained on them would score near-perfectly while
> learning nothing - and `Protocol` is text here, which a model cannot consume.
>
> For model training use `features_scaled.csv` (or `features_clean.csv`), which
> keep the numeric `Protocol_*` one-hot columns and exclude the identifiers
> entirely, and `labels.csv` for the encoded targets:
>
> ```python
> features = pd.read_csv(config.STEP4_OUTPUT_DIR / "features_scaled.csv")
> labels = pd.read_csv(config.STEP4_OUTPUT_DIR / "labels.csv")
> X = preprocessing.get_feature_matrix(features)   # identifiers and labels excluded
> y = labels["Label_encoded"]
> ```
>
> `config.ID_COLUMNS_NOT_FOR_TRAINING` names the columns that must never be fed
> to a model, and they are not in `feature_list.json`.

Set `config.COMBINED_ID_COLUMNS = []` for the features-and-labels-only layout,
or `COMBINED_USE_SCALED = False` to carry the unscaled features instead.


In [ ]:
import pandas as pd

combined_path = config.COMBINED_CSV
if not config.usable_output_dir(config.STEP4_OUTPUT_DIR) or not combined_path.exists():
    print("preprocessing.csv not found at", combined_path)
    print("Run the Step 4 cell above first.")
    final = None
else:
    final = pd.read_csv(combined_path, low_memory=False)
    size_mb = combined_path.stat().st_size / 1024 / 1024
    preprocessing.show_table(preprocessing.kv_table([
        ("absolute path", str(combined_path)),
        ("size (MB)", round(size_mb, 2)),
        ("rows", f"{final.shape[0]:,}"),
        ("columns", final.shape[1]),
        ("memory, deep (MB)", round(final.memory_usage(deep=True).sum() / 1024 / 1024, 1)),
    ]))

### Which columns are identifiers, features and labels

In [ ]:
if final is not None:
    ids = [c for c in config.COMBINED_ID_COLUMNS if c in final.columns]
    labels_cols = [c for c in config.COMBINED_LABEL_COLUMNS if c in final.columns]
    feature_cols = [c for c in final.columns if c not in ids + labels_cols]

    preprocessing.show_table(pd.DataFrame([
        {"role": "identifier (NOT a feature)", "columns": len(ids),
         "names": ", ".join(ids)},
        {"role": "feature (model input)", "columns": len(feature_cols),
         "names": ", ".join(feature_cols[:6]) + (", ..." if len(feature_cols) > 6 else "")},
        {"role": "label (target)", "columns": len(labels_cols),
         "names": ", ".join(labels_cols)},
        {"role": "TOTAL", "columns": final.shape[1], "names": ""},
    ]), note="`Protocol` is an identifier here, as readable text. The four numeric "
             "`Protocol_*` one-hot columns a model trains on are in "
             "`features_scaled.csv`, not in this file.")

    left_out = [c for c in config.COMBINED_DROP_FEATURE_COLUMNS if c in final.columns]
    extra = [c for c in ("Label_encoded", "Label_binary") if c in final.columns]
    print("[OK]   no Protocol_* one-hot columns" if not left_out
          else "[FAIL] found %s" % left_out)
    print("[OK]   no Label_encoded / Label_binary columns" if not extra
          else "[FAIL] found %s" % extra)

### head(10) and tail(5), all columns

In [ ]:
if final is not None:
    preprocessing.show_table(final.head(10), all_columns=True,
                             note="All %d columns; scroll sideways." % final.shape[1])
    preprocessing.show_table(final.tail(5), all_columns=True)

### Protocol counts

`Protocol` is the readable form of the four one-hot columns. The counts below
must match the one-hot sums in `features_scaled.csv` exactly - that is what the
verification table checks.


In [ ]:
if final is not None:
    counts = final["Protocol"].value_counts()
    rows = []
    for name in config.PROTOCOL_TEXT_VALUES:
        rows.append({"Protocol": name, "count": int(counts.get(name, 0)),
                     "%": round(100.0 * int(counts.get(name, 0)) / len(final), 3)})
    rows.append({"Protocol": "TOTAL", "count": len(final), "%": 100.0})
    preprocessing.show_table(pd.DataFrame(rows))

    unexpected = sorted(set(final["Protocol"].dropna().astype(str))
                        - set(config.PROTOCOL_TEXT_VALUES))
    blank = int(final["Protocol"].isna().sum())
    print("[OK]   Protocol holds only %s" % "/".join(config.PROTOCOL_TEXT_VALUES)
          if not unexpected else "[FAIL] unexpected values: %s" % unexpected)
    print("[OK]   no blank Protocol cells" if blank == 0
          else "[FAIL] %d blank Protocol cells" % blank)

### Label counts in the combined file

In [ ]:
if final is not None:
    counts = final["Label"].value_counts()
    table = pd.DataFrame({"Label": counts.index.astype(str),
                          "count": counts.to_numpy(),
                          "%": (100.0 * counts / len(final)).round(3).to_numpy()})
    table = pd.concat([table, pd.DataFrame([{"Label": "TOTAL", "count": len(final),
                                             "%": 100.0}])], ignore_index=True)
    preprocessing.show_table(table)

### Verification

Re-read from disk and checked against the other Step 4 files: the column order,
the absence of the `Protocol_*` and `Label_encoded`/`Label_binary` columns, the
identifier values against `metadata.csv` row by row, the `Protocol` counts
against the one-hot sums, the labels against `labels.csv`, and the feature values
against `features_scaled.csv`.


In [ ]:
if final is not None and "step4" in dir() and "features_scaled" in step4:
    table, ok, _ = preprocessing.verify_combined(
        final, step4["features_scaled"], step4["labels"], step4["metadata"],
        expected_rows=len(step4["features_scaled"]))
    print()
    print("ALL CHECKS PASSED:", ok)
elif final is not None:
    print("Step 4 was loaded from disk, so the in-memory frames are not available.")
    print("Re-run the Step 4 cell with OVERWRITE_STEP4 = True for the full check.")

## What Step 5 picks up

The Step 4 outputs are **unsplit and unbalanced on purpose**:

1. `features_clean.csv` is the one Step 5 splits, then fits a scaler on the
   training rows only - using `features_scaled.csv` across a split would leak
   test statistics into training.
2. The class imbalance is untouched, and Step 5 leaves it untouched too. Check
   `labels.csv`: the smallest classes have very few rows, so a stratified split
   and macro-F1 / per-class recall matter far more than accuracy.
3. `metadata.csv` holds `Src IP`, `Dst IP`, the ports and the timestamps. They
   stay **out of the feature matrix** - attacker IPs decided the labels in
   Step 1, so training on them is label leakage - but the file is kept, because
   the sequence work needs it and because it carries each row back to its flow.


---

# Step 5 - Train-Test Validation Split

A single stratified 80/20 split of the Step 4 outputs, and **nothing else**.

| Step 5 does | Step 5 does not |
|---|---|
| a stratified split on row **positions**, so features, labels and metadata stay aligned | handle the class imbalance - no SMOTE, no resampling, no class weights |
| refit the scaler on the **training rows only** | select features |
| report how much of the test set is feature-identical to training | train any model |

Three settings decide everything, all in `config.py`:

```python
TEST_SIZE       = 0.20      # 20% of the rows go to test
RANDOM_STATE    = 42        # fixed, so the split is reproducible
STRATIFY_COLUMN = "Label"   # the MULTICLASS label
OVERWRITE_STEP5 = False     # keep existing outputs instead of resplitting
```

**Why stratify on `Label` and not `Label_binary`.** Stratifying on the binary
target would balance attack-vs-benign while letting a 23-row class land entirely
on one side of the split - and a class with no test rows cannot be evaluated at
all. Stratifying on the multiclass label keeps every attack type on both sides.

**Why the split is on indices.** `train_test_split` is given `np.arange(n_rows)`,
not the frames. The three frames are then selected with the same positions, so
row *i* of `X_train`, `y_train` and `meta_train` is guaranteed to be the same
flow. Splitting each frame separately would be three independent shuffles.


### Step 5 settings

In [ ]:
from src.mscad import splitting

splitting.show_table(splitting.kv_table([
    ("input folder", str(config.STEP4_OUTPUT_DIR)),
    ("output folder", str(config.STEP5_OUTPUT_DIR)),
    ("TEST_SIZE", config.TEST_SIZE),
    ("RANDOM_STATE", config.RANDOM_STATE),
    ("STRATIFY_COLUMN", config.STRATIFY_COLUMN),
    ("SCALER (refitted on train)", config.SCALER),
    ("CLIP_TEST_TO_TRAIN_RANGE", config.CLIP_TEST_TO_TRAIN_RANGE),
    ("OVERWRITE_STEP5", config.OVERWRITE_STEP5),
    ("outputs already exist", splitting.step5_outputs_exist()),
]), note="With `OVERWRITE_STEP5 = False` and the outputs already present, they "
         "are loaded instead of resplit, so the split never changes under you.")

### Run Step 5

`run_step5()` does 5.0 input checks, 5.1 the split, 5.2 the refitted scaler,
5.3 the leakage count, then writes every file and re-reads it from disk.


In [ ]:
step5 = splitting.run_step5()

## 5.1 Split - class balance

The table below is the one that matters. `share drift pp` is how far a class's
share of the training rows differs from its share of the test rows, in
percentage points. A stratified split can only move **whole rows**, so the
smallest achievable drift is one test row's worth of share; the check uses
whichever is larger, that figure or `config.SHARE_TOLERANCE_PP`.


In [ ]:
splitting.show_table(step5["balance"],
                     note="Per class: counts and shares in the full dataset, train "
                          "and test. Saved as class_balance.csv, so it is shown "
                          "whether Step 5 just ran or was loaded from disk.")

### Binary target (benign vs attack)

In [ ]:
splitting.show_table(step5["binary_balance"],
                     note="`Label_binary` is 0 for Normal/Benign and 1 for every "
                          "attack. It is NOT what the split was stratified on -- it "
                          "is shown to confirm the binary target came out balanced "
                          "as well.")

### Class counts, train vs test

Log scale, because the classes differ by four orders of magnitude. Only the test
bars carry a number: the question this chart answers is whether every class still
has test rows.


In [ ]:
chart = config.STEP5_OUTPUT_DIR / config.STEP5_FILES["chart"]
if chart.exists():
    if splitting.dl.in_notebook():
        from IPython.display import Image, display
        display(Image(filename=str(chart)))
    else:
        print("chart saved to", chart)
else:
    print("No chart at", chart)
    print("Install matplotlib and re-run Step 5 with OVERWRITE_STEP5 = True.")

## 5.2 Scale - fitted on the training rows only

Step 4's scaler saw every row, test rows included, so its min/max carry
information from the test set. Step 5 therefore starts from the **unscaled**
`features_clean.csv` and fits a fresh scaler on `X_train` alone.

Scaled test values outside `[0, 1]` are **expected**: the scaler never saw those
rows, so a test flow can be larger than every training flow. They are counted and
reported, not clipped - `CLIP_TEST_TO_TRAIN_RANGE = False`, because clipping
would quietly flatten real differences between large flows.

The one-hot `Protocol_*` columns and the `zero_duration` marker are left alone;
they are already 0/1.


In [ ]:
if "scale_info" in step5:
    info = step5["scale_info"]
    splitting.show_table(splitting.kv_table([
        ("scaler", info["scaler_name"]),
        ("fitted on", "X_train only (%s rows)" % f"{len(step5['X_train']):,}"),
        ("columns scaled", "%d of %d" % (info["n_columns"], step5["X_train"].shape[1])),
        ("columns left alone", ", ".join(
            c for c in step5["X_train"].columns if c not in set(info["columns"]))),
        ("train range", "[%.6f, %.6f]" % (info["train_min"], info["train_max"])),
        ("test range", "[%.6f, %.6f]" % (info["test_min"], info["test_max"])),
        ("test cells outside [0, 1]", f"{info.get('test_cells_outside_0_1', 0):,}"),
        ("clipped", "no" if not info["clipped"] else "yes"),
        ("saved scaler", str(config.STEP5_OUTPUT_DIR / config.STEP5_FILES["scaler"])),
    ]))
    if len(info["out_of_range"]):
        splitting.show_table(info["out_of_range"].head(20),
                             note="Per column: how many test values fall outside "
                                  "[0, 1], and how far. Expected and small.")
else:
    # the loaded path: the same numbers, read back from split_summary.csv
    splitting.show_table(step5["split_summary"],
                         note="Read from split_summary.csv -- Step 5 was loaded from "
                              "disk, so there is no in-memory scaler report. Set "
                              "`OVERWRITE_STEP5 = True` to recompute it.")

## 5.3 Leakage check - reported, nothing dropped

About half the rows in this dataset share identical feature values once the
identifiers are removed: thousands of scan, flood and brute-force flows genuinely
look alike. A stratified random split therefore puts feature-identical twins on
both sides.

> **This inflates every test score and belongs in the results.** A model can
> "recognise" a test row it has effectively already seen in training, so the
> headline accuracy is optimistic. Report this percentage next to it, and lean on
> per-class recall for the classes whose share here is **low** - those are the
> rows the model actually had to generalise to.
>
> Nothing is dropped. These are real flows, and deleting them would change the
> dataset and break the attack-sequence work that comes later.


In [ ]:
splitting.show_table(step5["leakage_table"],
                     note="An exact match on the full UNSCALED feature vector. Saved "
                          "as leakage.csv.")

total = step5["leakage_table"].iloc[-1]
print("%s of %s test rows (%s%%) have a feature-identical twin in training."
      % (f"{int(total['also in train']):,}", f"{int(total['test rows']):,}",
         total["% of this class's test rows"]))

## 5.4 Saved files

Everything below is in `C:\MSCAD2022\outputs\step5\`.

| file | what it is |
|---|---|
| `X_train.csv` / `X_test.csv` | the features, scaled with the **train-fitted** scaler - this is what a model trains and is evaluated on |
| `X_train_unscaled.csv` / `X_test_unscaled.csv` | the same rows straight from `features_clean.csv`, for a model that wants its own scaling |
| `y_train.csv` / `y_test.csv` | `Label`, `Label_encoded`, `Label_binary` |
| `meta_train.csv` / `meta_test.csv` | the identifiers, including `Orig_Row`, so any row goes back to its flow |
| `train_indices.csv` / `test_indices.csv` | the row positions in the Step 4 files - the split, reproducible without rerunning it |
| `train_view.csv` / `test_view.csv` | the `preprocessing.csv` layout, **for viewing only** |
| `scaler_train.joblib` | the fitted scaler, with the settings it was fitted under |
| `split_summary.csv`, `step5_report.csv` | the numbers and the actions |
| `split_class_counts.png` | the chart above |

`X_train.csv` is the training matrix: the identifiers are not in it, and the
`Protocol_*` one-hot columns are. The two `*_view.csv` files are the readable
counterpart and must never be fed to a model.


In [ ]:
import pandas as pd

folder = config.usable_output_dir(config.STEP5_OUTPUT_DIR)
if folder is None:
    print("Step 5 outputs are not readable here:",
          config.why_unusable(config.STEP5_OUTPUT_DIR))
else:
    rows = []
    for key, name in config.STEP5_FILES.items():
        path = folder / name
        rows.append({"file": name, "exists": path.exists(),
                     "size (MB)": round(path.stat().st_size / 1024 / 1024, 3)
                                  if path.exists() else 0.0,
                     "what": key})
    splitting.show_table(pd.DataFrame(rows), note="Folder: %s" % folder)

### Shapes, and the head of each side

In [ ]:
X_train = pd.read_csv(config.STEP5_OUTPUT_DIR / config.STEP5_FILES["X_train"],
                      low_memory=False)
X_test = pd.read_csv(config.STEP5_OUTPUT_DIR / config.STEP5_FILES["X_test"],
                     low_memory=False)
y_train = pd.read_csv(config.STEP5_OUTPUT_DIR / config.STEP5_FILES["y_train"])
y_test = pd.read_csv(config.STEP5_OUTPUT_DIR / config.STEP5_FILES["y_test"])

splitting.show_table(splitting.kv_table([
    ("X_train", "%s rows x %d columns" % (f"{X_train.shape[0]:,}", X_train.shape[1])),
    ("X_test", "%s rows x %d columns" % (f"{X_test.shape[0]:,}", X_test.shape[1])),
    ("y_train", "%s rows x %d columns" % (f"{y_train.shape[0]:,}", y_train.shape[1])),
    ("y_test", "%s rows x %d columns" % (f"{y_test.shape[0]:,}", y_test.shape[1])),
    ("train + test", f"{len(X_train) + len(X_test):,}"),
    ("identical columns", list(X_train.columns) == list(X_test.columns)),
]))

counts = y_test[config.LABEL_COLUMN].value_counts()
splitting.show_table(pd.DataFrame({"Label": counts.index.astype(str),
                                   "test rows": counts.to_numpy()}),
                     note="Per-class test counts -- every class must appear here.")
splitting.show_table(X_train.head(5), all_columns=True,
                     note="X_train.head(5), all %d columns." % X_train.shape[1])

### Verification

Every file above was re-read from disk and checked: the row counts add up to the
Step 4 total, no row position is in both sides, `X_train` and `X_test` have the
same columns in the same order, the labels and metadata line up with the split
positions (compared on `Orig_Row`), and neither matrix holds a NaN or an inf.


In [ ]:
if "saved" in step5 and "verification" in step5.get("saved", {}):
    splitting.show_table(step5["saved"]["verification"])
    print()
    print("ALL CHECKS PASSED:", step5["saved"]["ok"])
else:
    print("Step 5 was loaded from disk, so there is no fresh verification table.")
    print("Set OVERWRITE_STEP5 = True and re-run to split and verify again.")

## What Step 6 picks up

1. **Train on `X_train.csv` only.** `X_test.csv` is touched once, at the end, to
   report a score. Any choice made by looking at the test set - a threshold, a
   feature, a hyper-parameter - makes that score meaningless.
2. **The imbalance is still there, on purpose.** SMOTE, resampling or class
   weights belong in Step 6 and must be fitted on the **training** side of this
   split only. Oversampling before the split copies rows across it, which is the
   same leak as scaling before the split.
3. **Use `train_indices.csv` / `test_indices.csv`** to carry this exact split
   into any other experiment, instead of resplitting with the same seed and
   hoping.
4. **Report the 5.3 leakage percentage next to every score.** Macro-F1 and
   per-class recall, not accuracy: with Brute_Force at tens of thousands of rows
   and Web_Crwling at 23, accuracy is nearly a measure of the majority class.


---
# Stage 3 - preprocessing: clean, encode, scale

_(no code yet)_

---
# Stage 4 - feature selection (Mutual Information / Permutation Importance)

_(no code yet)_

---
# Stage 5 - autoencoder feature extraction

_(no code yet)_

---
# Stage 6 - class-imbalance handling

_(no code yet)_

---
# Stage 7 - models: RF, XGBoost, LSTM, CNN, hybrid CNN-LSTM

_(no code yet)_

---
# Stage 8 - evaluation, sliding windows, PrefixSpan, attack-state graph

This is why Step 1 and Step 2 keep Src IP / Dst IP / ports / protocol / Timestamp.

_(no code yet)_